<a href="https://colab.research.google.com/github/Mehrun-c/GlobusSoft/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import io
import cv2
import numpy as np
import pickle
import os
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Optional

In [2]:
app = FastAPI(
    title="Face Authentication API",
    description="Verify whether two face images belong to the same person using HOG + SVM",
    version="1.0.0"
)

MODEL_PATH = "face_auth_model.pkl"
HAAR_PATH  = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
FACE_CASCADE = cv2.CascadeClassifier(HAAR_PATH)
THRESHOLD  = 0.72   # cosine similarity threshold


In [3]:
model_data = None

@app.on_event("startup")
def load_model():
    global model_data
    if os.path.exists(MODEL_PATH):
        with open(MODEL_PATH, "rb") as f:
            model_data = pickle.load(f)
        print(f"[INFO] Model loaded from '{MODEL_PATH}'")
    else:
        print(f"[WARNING] Model file '{MODEL_PATH}' not found. "
              "Run task2_train.py first, then restart the server.")


/tmp/ipykernel_10302/3675665685.py:3: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [39]:
def read_image_from_bytes(data: bytes) -> np.ndarray:
    arr = np.frombuffer(data, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    return img


def detect_and_crop_face(image_bgr: np.ndarray):
    gray  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    faces = FACE_CASCADE.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )
    if len(faces) == 0:
        return None, None
    faces = sorted(faces, key=lambda r: r[2]*r[3], reverse=True)
    x, y, w, h = faces[0]
    crop = gray[y:y+h, x:x+w]
    # Resize to the dimension (Width, Height) that produces 5940 HOG features
    # Corresponding to image shape (120, 88) (Height, Width)
    resized = cv2.resize(crop, (88, 120))
    return resized, {"x": int(x), "y": int(y), "w": int(w), "h": int(h)}


def extract_hog(image_gray: np.ndarray) -> np.ndarray:
    # HOG parameters for a (120, 88) image (H, W) to produce 5940 features
    # _winSize takes (width, height)
    hog = cv2.HOGDescriptor(
        _winSize   =(88, 120),
        _blockSize =(8, 8),
        _blockStride=(8, 8), # Changed blockStride to (8,8) for 5940 features
        _cellSize  =(4, 4),
        _nbins     =9
    )
    if image_gray.max() <= 1.0:
        image_gray = (image_gray * 255).astype(np.uint8)
    return hog.compute(image_gray).flatten()


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom != 0 else 0.0

In [41]:
print(len((face1)))

64


In [42]:
class VerifyResponse(BaseModel):
    verification_result: str
    similarity_score   : float
    identity_image1    : str
    identity_image2    : str
    bbox_image1        : Optional[dict]
    bbox_image2        : Optional[dict]


In [43]:
@app.get("/")
def root():
    return {"message": "Face Authentication API is running. POST two images to /verify"}


@app.get("/health")
def health():
    return {
        "status"      : "ok",
        "model_loaded": model_data is not None
    }


@app.post("/verify", response_model=VerifyResponse)
async def verify(
    image1: UploadFile = File(..., description="First face image"),
    image2: UploadFile = File(..., description="Second face image"),
):
    """
    Upload two face images and get a same/different-person verdict.

    - **image1**: first face image (jpg/png)
    - **image2**: second face image (jpg/png)
    """
    if model_data is None:
        raise HTTPException(
            status_code=503,
            detail="Model not loaded. Run task2_train.py first and restart the server."
        )

    pipeline     = model_data["pipeline"]
    target_names = model_data["target_names"]
    img_shape    = model_data["image_shape"]   # (H, W)

    # Read uploaded images
    raw1 = await image1.read()
    raw2 = await image2.read()
    img1 = read_image_from_bytes(raw1)
    img2 = read_image_from_bytes(raw2)

    if img1 is None:
        raise HTTPException(status_code=400, detail="Cannot decode image1.")
    if img2 is None:
        raise HTTPException(status_code=400, detail="Cannot decode image2.")

    # Detect faces
    face1, bbox1 = detect_and_crop_face(img1, img_shape)
    face2, bbox2 = detect_and_crop_face(img2, img_shape)

    if face1 is None:
        raise HTTPException(status_code=422, detail="No face detected in image1.")
    if face2 is None:
        raise HTTPException(status_code=422, detail="No face detected in image2.")

    # Feature extraction
    feat1 = extract_hog(face1).reshape(1, -1)
    feat2 = extract_hog(face2).reshape(1, -1)

    # Predict identities
    pred1 = pipeline.predict(feat1)[0]
    pred2 = pipeline.predict(feat2)[0]
    name1 = target_names[pred1]
    name2 = target_names[pred2]

    # Cosine similarity on scaled features
    scaler  = pipeline.named_steps["scaler"]
    scaled1 = scaler.transform(feat1).flatten()
    scaled2 = scaler.transform(feat2).flatten()
    sim     = cosine_similarity(scaled1, scaled2)

    # Decision
    is_same = (pred1 == pred2) and (sim >= THRESHOLD)
    verdict = "same person" if is_same else "different person"

    return VerifyResponse(
        verification_result=verdict,
        similarity_score   =round(sim, 4),
        identity_image1    =name1,
        identity_image2    =name2,
        bbox_image1        =bbox1,
        bbox_image2        =bbox2,
    )

In [47]:
# ───────── SIMPLE TESTING CODE ─────────

IMG1 = "//content//face1.jpg.jpeg"
IMG2 = "//content//face2.jpg.jpeg"

# Load model
load_model()

# Read images
img1 = cv2.imread(IMG1)
img2 = cv2.imread(IMG2)

# Detect faces
face1, bbox1 = detect_and_crop_face(img1) # Removed model_data["image_shape"] as target_size is no longer needed
face2, bbox2 = detect_and_crop_face(img2) # Removed model_data["image_shape"] as target_size is no longer needed

# Extract features
feat1 = extract_hog(face1).reshape(1, -1)
feat2 = extract_hog(face2).reshape(1, -1)

# Predict identities
pipeline = model_data["pipeline"]
target_names = model_data["target_names"]

pred1 = pipeline.predict(feat1)[0]
pred2 = pipeline.predict(feat2)[0]

name1 = target_names[pred1]
name2 = target_names[pred2]

# Similarity
scaler = pipeline.named_steps["scaler"]

scaled1 = scaler.transform(feat1).flatten()
scaled2 = scaler.transform(feat2).flatten()

sim = cosine_similarity(scaled1, scaled2)

# Final result
if pred1 == pred2 and sim >= THRESHOLD:
    verdict = "same person"
else:
    verdict = "different person"

# PRINT OUTPUT
print("\n===== FACE VERIFICATION RESULT =====\n")

print("Verification Result :", verdict)
print("Similarity Score    :", round(sim, 4))
print("Identity Image1     :", name1)
print("Identity Image2     :", name2)
print("Bounding Box Image1 :", bbox1)
print("Bounding Box Image2 :", bbox2)

[INFO] Model loaded from 'face_auth_model.pkl'

===== FACE VERIFICATION RESULT =====

Verification Result : different person
Similarity Score    : 0.3596
Identity Image1     : George W Bush
Identity Image2     : George W Bush
Bounding Box Image1 : {'x': 184, 'y': 65, 'w': 142, 'h': 142}
Bounding Box Image2 : {'x': 231, 'y': 69, 'w': 293, 'h': 293}
